In [9]:
# Standard Libraies
from typing import Annotated, Sequence, TypedDict, Literal
from dotenv import load_dotenv

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END


class NodeAInput(TypedDict):
    chat_input: str


class NodeAOutput(TypedDict):
    node_a_output: str


class NodeBInput(TypedDict):
    node_b_input: str


class NodeBOutput(TypedDict):
    node_b_output: str


class NodeCInput(TypedDict):
    node_a_output: str


class NodeCOutput(TypedDict):
    node_c_output: int

class GraphState(TypedDict):
    chat_input: str
    chat_output: int


def node_a(state: NodeAInput) -> NodeAOutput:
    return {'node_a_output': 3}


def node_b(state: NodeBInput) -> NodeBOutput:
    return {'node_b_output': 'done'}

def node_c(state: NodeCInput) -> GraphState:
    chat_output = int(state['node_a_output'] + 1)

    return {'chat_output': chat_output}



workflow = (
    StateGraph(GraphState)
        .add_node("node_a", node_a)
        .add_node("node_b", node_b)
        .add_node("node_c", node_c)

        .add_edge(START, "node_a")
        .add_edge("node_a", "node_b")
        .add_edge("node_b", "node_c")
        .add_edge("node_c", END)

        .compile()
)

result = workflow.invoke(input={'chat_input': 'Hi'})
result

{'chat_input': 'Hi', 'chat_output': 4}